The MergerRetriever and Long Context Reorder are advanced RAG techniques designed to solve 
two specific problems: 
 1. "Data Silos" (where information is spread across different databases).
 2. "Lost in the Middle" (where LLMs ignore information buried in the center of a long prompt).

**MergerRetriever (The "Lord of the Retrievers")**
The MergerRetriever (often nicknamed LOTR) is used when you have multiple vector stores or 
different types of retrievers (e.g., one for technical manuals, one for general FAQs) and 
you want to combine their results into one unified list.

**Why do you need it?**
   - Diverse Sources: You might have data in ChromaDB, FAISS, and a SQL database.
   - Different Embeddings: One retriever might use OpenAIEmbeddings for semantic search, 
     while another uses HuggingFaceEmbeddings for specific technical terminology.
   - Improved Recall: Using multiple retrievers reduces the risk that any single algorithm misses the "perfect" document.

**How it works:**

It takes a list of retrievers and executes them all. It then merges their outputs. By default, it does a round-robin merge: it takes the 1st result from Retriever A, then the 1st from Retriever B, then the 2nd from A, and so on.

In [1]:
from langchain_classic.document_loaders import PyPDFLoader
document_harrypotter = PyPDFLoader("Data/harrypotter.pdf").load()

In [2]:
document_got = PyPDFLoader("Data/GOT.pdf").load()

In [3]:
len(document_harrypotter)

3623

In [ ]:
# Lets create splitters for chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 100,
    length_function = len,
    add_start_index = True # tracks where the chunk starts in the original doc
)

In [5]:
chunks_got = text_splitter.split_documents(document_got)
chunks_harrypotter = text_splitter.split_documents(document_harrypotter)

In [6]:
len(chunks_got)

4295

In [7]:
len(chunks_harrypotter)

17237

In [15]:
# Define the embedding models
from langchain_ollama import OllamaEmbeddings

embedding_nomic = OllamaEmbeddings(
    model = "nomic-embed-text"
)
embedding_gemma = OllamaEmbeddings(
    model = "embeddinggemma:300m"
)

In [9]:
# now create embeddings and ingest them inside chroma
from langchain_classic.vectorstores import Chroma
import chromadb

In [10]:
client_settings = chromadb.config.Settings(
    is_persistent = True,
    persist_directory = "chroma_db",
    anonymized_telemetry = False,
)

In [11]:
harrypotter_vectorstore = Chroma.from_documents(
    documents = chunks_harrypotter,
    client_settings = client_settings,
    embedding = embedding_nomic,
    collection_name = "harrypotter",
    collection_metadata={"hnsw":"cosine"},
    persist_directory = "./store/harrypotter"
)

In [16]:
got_vectorstore = Chroma.from_documents(
    documents = chunks_got,
    client_settings = client_settings,
    embedding = embedding_gemma,
    collection_name = "got",
    collection_metadata={"hnsw":"cosine"},
    persist_directory = "./store/got"
)

In [17]:
# get the retrievers
harrypotter_retriever = harrypotter_vectorstore.as_retriever(
    search_kwargs = {"k" : 5}
)
got_retriever = got_vectorstore.as_retriever(
    search_kwargs = {"k" : 5}
)

In [18]:
# Define Lotr(Merge Retriever)
from langchain_classic.retrievers import MergerRetriever
lotr = MergerRetriever(
    retrievers = [harrypotter_retriever , got_retriever]
)

In [22]:
for chunks in lotr.invoke("Who was the jon snow?"):
    print(chunks.page_content)
    print(" = " * 25)

.
“What’s	going	on	here?”	came	the	cold,	drawling	voice	of	Draco	Malfoy.
Harry	started	stuffing	everything	feverishly	into	his	ripped	bag,	desperate	to
get	away	before	Malfoy	could	hear	his	musical	valentine.
“What’s	all	this	commotion?”	said	another	familiar	voice	as	Percy	Weasley
arrived.
Losing	his	head,	Harry	tried	to	make	a	run	for	it,	but	the	dwarf	seized	him
around	the	knees	and	brought	him	crashing	to	the	floor.
“Right,”	he	said,	sitting	on	Harry’s	ankles
 =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  = 
.
Tyrion Lannister had claimed that most men would rather deny a hard truth than face it, 
but Jon was done with denials. He was who he was; Jon Snow, bastard and oathbreaker, 
motherless, friendless, and damned. For the rest of his life—however long that might be
—he would be condemned to be an outsider, the silent man standing in the shadows who 
dares not speak his true name
 =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  = 

In [23]:
for chunks in lotr.invoke("Who is a harry potter?"):
    print(chunks.page_content)
    print(" = " * 25)

.	Everyone	from	wizarding
 =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  = 
:
 =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  = 
.	
“Are
you	a	wizard,	or	what?”
“Oh	—	right	—	yeah	—”
Ron	looked	around,	then	directed	his	wand	at	a	twig	on	the	ground	and
 =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  = 
. Where is the boy? Tyrion wondered.
A warhorn blew. Haroooooooooooooooooooooooo, it cried, its voice as long and low and
 =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  = 
.	 We	 did	 Animagi	 in	 class	 with	 Professor	 McGonagall.	 And	 I
looked	them	up	when	I	did	my	homework	—	the	Ministry	of	Magic	keeps
tabs	 on	 witches	 and	 wizards	 who	 can	 become	 animals;	 there’s	 a	 register
showing	what	animal	they	become,	and	their	markings	and	things	.	.
 =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  =  = 
. The Lannister words are Hear Me 
Roar
 =  =  =  =  =  =  =

In [24]:
# now to fix this randomness we need to add a filter after the merger.
# The filter will act like a judge that looks  at the combined list and removes anything
# that isn't actually relevant to the query.

from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import DocumentCompressorPipeline
from langchain_community.document_transformers import EmbeddingsRedundantFilter

In [25]:
# choose a Judge embedding(a llm which will jutify that doc is relevant or not)
filter_embedding = embedding_nomic # also can take other models

# define redundant filter
# (If we have duplicate context in both candidate docs then keep one and remove others)
redundant_filter = EmbeddingsRedundantFilter(embeddings = filter_embedding)

# here also we can add a Embedding Filtering to remove irrelavant

# define re-ordering to fix Long Context
from langchain_community.document_transformers.long_context_reorder import LongContextReorder
reordering = LongContextReorder()

# Now merge all these in a single pipeline
pipeline = DocumentCompressorPipeline(
    transformers = [redundant_filter , reordering]
)

# now create the final retriever
compression_retriever = ContextualCompressionRetriever(
    base_compressor = pipeline,
    base_retriever = lotr
)

In [31]:
res = compression_retriever.invoke(input = "Who was the jon snow?")

In [35]:
for i , doc in enumerate(res):
    print(f"------- Document {i} ------------")
    print(doc.page_content)
    print("\n")

------- Document 0 ------------
.
Tyrion Lannister had claimed that most men would rather deny a hard truth than face it, 
but Jon was done with denials. He was who he was; Jon Snow, bastard and oathbreaker, 
motherless, friendless, and damned. For the rest of his life—however long that might be
—he would be condemned to be an outsider, the silent man standing in the shadows who 
dares not speak his true name


------- Document 1 ------------
. “Hello, Ghost.”
Jon Snow moved closer. He looked bigger and heavier in his layers of fur and leather, 
the hood of his cloak pulled down over his face. “Lannister,” he said, yanking loose the 
scarf to uncover his mouth. “This is the last place I would have expected to see you.” He 
carried a heavy spear tipped in iron, taller than he was, and a sword hung at his side in a 
leather sheath. Across his chest was a gleaming black warhorn, banded with silver


------- Document 2 ------------
. . . but I do not care to wake every dawn wondering if yo

In [29]:
# Define the system prompt
system_prompt = (
    "You are an question answering assistant."
    "Provide answers of the users question based on the given context."
    "If you don't know the answer or the given context is not sufficient "
    "then instead of hallucinating simply say you don't know."
    "Here is the context:\n\n{context}"
)
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_messages([
    ("system" , system_prompt),
    ("human" , "{input}") 
])

In [36]:
# load the chat model
from langchain_ollama import ChatOllama
llm = ChatOllama(
    model = "smollm2:1.7b",
    temperature = 0.1
)

In [37]:
# create stuff chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
combine_docs_chain = create_stuff_documents_chain(
    llm = llm , prompt = prompt
)

In [38]:
# create the final rag chain
from langchain_classic.chains import create_retrieval_chain

rag_chain = create_retrieval_chain(compression_retriever , combine_docs_chain)

In [45]:
query = "How many children Stark has?"
response = rag_chain.invoke({
    "input" : query
})

In [46]:
print(response['answer'])

Lord Stark has five trueborn children: three sons (Robb, Sansa, and Bran) and two daughters (Arya and Sansa).
